In [1]:
import pandas as pd

# Carregar o arquivo original
df = pd.read_parquet("03_PENSE2024_MICRODADOS_COM_LABELS.parquet")

# Exibir dimensões originais
print("Dimensões originais:", df.shape)

# Remover linhas com um ou mais valores faltantes
df_limpo = df.dropna()

# Garantir que todas as variáveis continuem como string após a remoção dos NaN
for coluna in df_limpo.columns:
    df_limpo[coluna] = df_limpo[coluna].astype("string")

# Exibir dimensões após a limpeza
print("Dimensões após limpeza:", df_limpo.shape)

# Calcular quantidade de linhas removidas
print("Linhas removidas:", len(df) - len(df_limpo))

# Salvar em um novo arquivo Parquet
df_limpo.to_parquet("04_PENSE2024_SEM_NAN.parquet", index=False)

print("Arquivo salvo com sucesso!")

Dimensões originais: (153870, 39)
Dimensões após limpeza: (148736, 39)
Linhas removidas: 5134
Arquivo salvo com sucesso!


In [2]:
import pandas as pd

# ============================================================
# 1. CARREGA AS BASES DE DADOS
# ============================================================

df_dict = pd.read_excel(
    '00_Dicionário_Microdados_PENSE2024.xlsx',
    engine='openpyxl'
)

df = pd.read_parquet(
    '04_PENSE2024_SEM_NAN.parquet'
)

# ============================================================
# 2. FUNÇÃO PARA NORMALIZAR TEXTOS
# ============================================================

def normalizar(texto):
    return "".join(str(texto).split()).lower()


# ============================================================
# 3. CRIA O MAPA DE CATEGORIAS
#
# Chave:
#     descrição da variável
#
# Valor:
#     {código da resposta: descrição da resposta}
# ============================================================

mapeamento = {}

for _, row in df_dict.iterrows():

    pergunta_desc = str(
        row['Descrição da Variável']
    ).strip()

    # Ignora variáveis sem descrição
    if (
        not pergunta_desc
        or pergunta_desc == 'nan'
    ):
        continue

    chave = normalizar(pergunta_desc)

    categorias = str(row['Categoria'])

    if categorias and categorias != 'nan':

        sub_mapa = {}

        for linha in categorias.split('\n'):

            if ' - ' in linha:

                cod, desc = linha.split(
                    ' - ',
                    1
                )

                cod = cod.strip()
                desc = desc.strip()

                # Guarda o código original
                sub_mapa[cod] = desc

                # Exemplo:
                # 1.0 -> 1
                try:
                    sub_mapa[
                        str(int(float(cod)))
                    ] = desc
                except (ValueError, TypeError):
                    pass

                # Exemplo:
                # 1 -> 1.0
                try:
                    sub_mapa[
                        str(float(cod))
                    ] = desc
                except (ValueError, TypeError):
                    pass

        mapeamento[chave] = sub_mapa


# ============================================================
# 4. PROCESSA CADA COLUNA DO DATAFRAME
# ============================================================

linhas_finais = []

for col in df.columns:

    # --------------------------------------------------------
    # Label completo da coluna
    #
    # Exemplo:
    #
    # B01001A - Qual é o seu sexo?
    # --------------------------------------------------------

    col_limpa = str(col).strip()

    # --------------------------------------------------------
    # SEPARA CÓDIGO E DESCRIÇÃO
    # --------------------------------------------------------

    if ' - ' in col_limpa:

        codigo_coluna, pergunta_desc = (
            col_limpa.split(' - ', 1)
        )

        codigo_coluna = codigo_coluna.strip()
        pergunta_desc = pergunta_desc.strip()

    else:

        codigo_coluna = col_limpa
        pergunta_desc = col_limpa

    # --------------------------------------------------------
    # PROCURA A PERGUNTA PELO SEU TEXTO
    # --------------------------------------------------------

    chave_busca = normalizar(
        pergunta_desc
    )

    mapa_col = mapeamento.get(
        chave_busca,
        {}
    )

    # --------------------------------------------------------
    # DISTRIBUIÇÃO DOS VALORES
    # --------------------------------------------------------

    dist = (
        df[col]
        .value_counts(
            normalize=True,
            dropna=False
        )
        * 100
    )

    linhas_respostas = []

    for val, pct in dist.items():

        # ----------------------------------------------------
        # Valores NaN
        # ----------------------------------------------------

        if pd.isna(val):

            desc = 'Valores Nulos / Em Branco (NaN)'

        else:

            v_str = str(val).strip()

            # Corrige 1.0 -> 1
            if v_str.endswith('.0'):
                v_str = v_str[:-2]

            desc = mapa_col.get(
                v_str,
                'Significado não encontrado'
            )

        linhas_respostas.append(
            f'[{val}] {desc}: {pct:.2f}%'
        )

    # --------------------------------------------------------
    # Adiciona resultado da pergunta
    # --------------------------------------------------------

    linhas_finais.append({
        'Pergunta / Variável': col_limpa,
        'Distribuição das Respostas':
            '\n'.join(linhas_respostas)
    })


# ============================================================
# 5. CRIA A TABELA FINAL
# ============================================================

df_consolidado = pd.DataFrame(
    linhas_finais
)


# ============================================================
# 6. ENVIA PARA O CLIPBOARD
# ============================================================

df_consolidado.to_clipboard(
    index=False,
    sep='\t'
)


# ============================================================
# 7. EXIBE A TABELA NO NOTEBOOK
# ============================================================

print(
    "✨ Dados copiados para a área de transferência!"
)

print()

print(
    "📊 EXIBIÇÃO DOS RESULTADOS NO IPYNB:"
)

display(
    df_consolidado.style.set_properties(
        **{
            'white-space': 'pre-wrap',
            'text-align': 'left'
        }
    )
)

✨ Dados copiados para a área de transferência!

📊 EXIBIÇÃO DOS RESULTADOS NO IPYNB:


,Pergunta / Variável,Distribuição das Respostas
0,CAPITAL - indica se é município da capital,[1] é capital: 51.21% [0] não é capital: 48.79%
1,B01001A - Qual é o seu sexo?,[2] Mulher: 50.05% [1] Homem: 49.95%
2,"IDADE_AGREG - IDADE DO ESTUDANTE, CATEGORIAS",[2] 13 a 15 anos: 51.62% [3] 16 e 17 anos: 27.75% [1] Menos de 13 anos: 16.03% [4] 18 anos ou mais: 4.52% [-9] Sem resposta: 0.08%
3,B01002 - Qual é a sua cor ou raça?,[4] Parda: 43.23% [1] Branca: 40.53% [2] Preta: 10.67% [3] Amarela: 2.75% [5] Indígena: 2.25% [-9] Sem resposta: 0.57% [-2] Abandono: 0.01%
4,B01006 - Você mora com sua mãe?,[1] Sim: 89.93% [2] Não: 10.02% [-9] Sem resposta: 0.03% [-2] Abandono: 0.01%
5,B01007 - Você mora com seu pai?,[1] Sim: 63.45% [2] Não: 36.47% [-9] Sem resposta: 0.07% [-2] Abandono: 0.02%
6,B01015B - Na sua casa tem computador ou notebook?,[1] Sim: 63.27% [2] Não: 36.67% [-9] Sem resposta: 0.04% [-2] Abandono: 0.02%
7,B01016 - Você tem acesso à internet em sua casa?,[1] Sim: 97.26% [2] Não: 2.69% [-9] Sem resposta: 0.03% [-2] Abandono: 0.02%
8,"B03006B - NOS ÚLTIMOS 7 DIAS, sem contar as aulas de educação física da escola, em quantos dias você praticou alguma atividade física?",[1] Nenhum dia nos últimos 7 dias: 28.98% [3] 2 dias: 13.02% [4] 3 dias: 12.34% [6] 5 dias: 12.12% [2] 1 dia: 10.64% [8] 5 dias mais sábado e domingo: 9.37% [5] 4 dias: 8.01% [7] 5 dias mais sábado: 5.22% [-9] Sem resposta: 0.23% [-2] Abandono: 0.08%
9,"B03010C - Quantas horas por dia você costuma ficar sentado(a), assistindo televisão, jogando videogame, usando computador, celular, tablet ou fazendo outras atividades sentado(a)? (NÃO contar sábado, domingo, feriados ou o tempo sentado na escola)",[1] Até 1 hora por dia: 14.39% [3] Mais de 2 horas até 3 horas por dia: 13.92% [2] Mais de 1 hora até 2 horas por dia: 13.60% [4] Mais de 3 horas até 4 horas por dia: 12.36% [10] Não sei / não lembro: 10.11% [5] Mais de 4 horas até 5 horas por dia: 9.83% [9] Mais de 8 horas por dia: 8.82% [6] Mais de 5 horas até 6 horas por dia: 7.64% [7] Mais de 6 horas até 7 horas por dia: 5.07% [8] Mais de 7 horas até 8 horas por dia: 3.86% [-9] Sem resposta: 0.31% [-2] Abandono: 0.09%
